# OWL Inference on LUBM Ontology with owlapy

This notebook uses [owlapy](https://github.com/dice-group/owlapy) to load the **univ-bench.owl** ontology and perform symbolic inferences:
- TBox exploration (classes, properties, axioms)
- Reasoner-based class hierarchy inference
- Satisfiability checking
- Instance retrieval over ABox data (University0_0.owl)

In [ ]:
from owlapy.owl_ontology import SyncOntology
from owlapy.owl_reasoner import SyncReasoner
from owlapy.class_expression import OWLClass
from owlapy.iri import IRI
from owlapy.owl_axiom import OWLSubClassOfAxiom

ONTO_PATH = "univ-bench-inst.ttl"
BASE_IRI  = "http://swat.cse.lehigh.edu/onto/univ-bench.owl#"

print("owlapy imported successfully")

## 1. Load the Ontology

In [ ]:
onto = SyncOntology(ONTO_PATH)
print(f"Ontology loaded: {onto.get_ontology_id()}")

## 2. Explore Classes and Properties

In [ ]:
# Named classes in the signature
classes = sorted(onto.classes_in_signature(), key=lambda c: c.remainder)
print(f"Total classes: {len(classes)}\n")
for cls in classes:
    print(f"  {cls.remainder}")

In [ ]:
# Object properties
obj_props = sorted(onto.object_properties_in_signature(), key=lambda p: p.remainder)
print(f"Object properties ({len(obj_props)}):")
for p in obj_props:
    print(f"  {p.remainder}")

print()

# Data properties
data_props = sorted(onto.data_properties_in_signature(), key=lambda p: p)
print(f"Data properties ({len(data_props)}):")
for p in data_props:
    print("p = ", p.iri.get_remainder())
    # print(f"  {p.remainder}")

## 3. TBox Axioms

In [ ]:
reasoner = SyncReasoner(ontology=ONTO_PATH, reasoner="HermiT")
print("Consistency:", reasoner.has_consistent_ontology())

In [ ]:
# SubClassOf axioms: use reasoner.super_classes(direct=True) to avoid the
# owlapy mapper bug that crashes on intersectionOf with a single operand.
print('Stated SubClassOf axioms (direct=True):\n')
for cls in sorted(onto.classes_in_signature(), key=lambda c: c.remainder):
    for sup in sorted(reasoner.super_classes(cls, direct=True), key=lambda c: c.remainder):
        if sup.remainder not in ('Thing', 'Nothing'):
            print(f'  {cls.remainder:30s} ⊑  {sup.remainder}')

In [ ]:
# Object property domain and range — use per-property methods to avoid the mapper bug
from owlapy.owl_axiom import OWLObjectPropertyDomainAxiom, OWLObjectPropertyRangeAxiom

print('Object property domain/range axioms:\n')
for prop in sorted(onto.object_properties_in_signature(), key=lambda p: p.remainder):
    for axiom in onto.object_property_domain_axioms(prop):
        d = axiom.get_domain()
        if hasattr(d, 'remainder'):
            print(f'  domain({prop.remainder}) = {d.remainder}')
    for axiom in onto.object_property_range_axioms(prop):
        r = axiom.get_range()
        if hasattr(r, 'remainder'):
            print(f'  range({prop.remainder})  = {r.remainder}')

## 4. Reasoner Initialisation

We use the **HermiT** reasoner (OWL DL-complete). owlapy also supports Pellet, JFact, Openllet, ELK, Structural.

In [ ]:
reasoner = SyncReasoner(ontology=ONTO_PATH, reasoner="HermiT")
print("Consistency:", reasoner.has_consistent_ontology())

## 5. Class Hierarchy Inference

The reasoner computes the *inferred* hierarchy, which may go beyond the stated `rdfs:subClassOf` triples.

In [ ]:
def make_class(name: str) -> OWLClass:
    return OWLClass(IRI(BASE_IRI, name))

# All inferred subclasses of Faculty (direct=False → full transitive closure)
faculty = make_class("Faculty")
# reasoner.
inferred_subs = list(reasoner.sub_classes(faculty, direct=False))

print(f"All inferred subclasses of Faculty ({len(inferred_subs)}):")
for sc in sorted(inferred_subs, key=lambda c: c.remainder):
    print(f"  {sc.remainder}")
## only level2 (inferred) = inferred (direct = False) - direct

In [ ]:
# Direct subclasses only, si mette direct = False
direct_subs = list(reasoner.sub_classes(faculty, direct=True))
print(f"Direct subclasses of Faculty ({len(direct_subs)}):")
for sc in sorted(direct_subs, key=lambda c: c.remainder):
    print(f"  {sc.remainder}")

In [ ]:
# Inferred superclass chain of AssistantProfessor
asst_prof = make_class("AssistantProfessor")
supers = list(reasoner.super_classes(asst_prof, direct=False))

print("Inferred superclasses of AssistantProfessor:")
for sc in sorted(supers, key=lambda c: c.remainder):
    print(f"  {sc.remainder}")

In [ ]:
# Build and display the full inferred class hierarchy as a tree
from owlapy.class_expression import OWLThing

def print_hierarchy(cls: OWLClass, reasoner: SyncReasoner, indent: int = 0):
    label = cls.remainder if hasattr(cls, 'remainder') else str(cls)
    print("  " * indent + label)
    for sub in sorted(reasoner.sub_classes(cls, direct=True), key=lambda c: c.remainder):
        print_hierarchy(sub, reasoner, indent + 1)

print("Inferred class hierarchy rooted at owl:Thing:\n")
print_hierarchy(OWLThing, reasoner)

## 6. Satisfiability Checking

A class is *unsatisfiable* when the reasoner can prove no individual could belong to it (it is equivalent to `owl:Nothing`).

In [ ]:
# Check satisfiability for every named class
print(f"{'Class':<30} Satisfiable?")
print("-" * 45)
for cls in sorted(onto.classes_in_signature(), key=lambda c: c.remainder):
    sat = reasoner.is_satisfiable(cls)
    marker = "✓" if sat else "✗  ← UNSATISFIABLE"
    print(f"  {cls.remainder:<28} {marker}")

In [ ]:
# Bulk query: all unsatisfiable classes at once
unsat = list(reasoner.unsatisfiable_classes())
if unsat:
    print("Unsatisfiable classes:")
    for cls in unsat:
        print(f"  {cls.remainder}")
else:
    print("No unsatisfiable classes — ontology is coherent.")

## 7. Equivalent Classes

In [ ]:
for cls in sorted(onto.classes_in_signature(), key=lambda c: c.remainder):
    equiv = [e for e in reasoner.equivalent_classes(cls) if e != cls]
    if equiv:
        names = ", ".join(
            e.remainder if hasattr(e, 'remainder') else str(e) for e in equiv
        )
        print(f"  {cls.remainder} ≡ {names}")

## 8. Instance-level Inference (ABox — univ-bench-inst.ttl)

`univ-bench-inst.ttl` bundles TBox and ABox in a single file. It contains 5 individuals:
2 `GraduateCourse` individuals and 3 `GraduateStudent` individuals linked via `takesCourse`.
We reuse the `onto` and `reasoner` already initialised above — no extra file needed.

In [ ]:
# Individuals stated in the ontology (ABox)
individuals = list(onto.individuals_in_signature())
print(f'Individuals in univ-bench-inst.ttl ({len(individuals)}):\n')
for ind in sorted(individuals, key=lambda i: i.remainder):
    print(f'  {ind.remainder}')

In [ ]:
# Direct instances of GraduateStudent (asserted in the ABox)
grad_student = make_class('GraduateStudent')
direct_instances = list(reasoner.instances(grad_student, direct=True))

print(f'Direct instances of GraduateStudent ({len(direct_instances)}):')
for ind in sorted(direct_instances, key=lambda i: i.remainder):
    print(f'  {ind.remainder}')

In [ ]:
# Inferred instances of Student (direct=False)
# HermiT infers that each GraduateStudent is also a Student because:
#   Student ≡ Person ∩ ∃takesCourse.Course
#   graduate_student_0x : GraduateStudent ⊑ Person, and takesCourse a GraduateCourse ⊑ Course
student = make_class('Student')
inferred_students = list(reasoner.instances(student, direct=False))
direct_students   = list(reasoner.instances(student, direct=True))

print(f'Inferred instances of Student (direct=False): {len(inferred_students)}')
print(f'Direct  instances of Student (direct=True):  {len(direct_students)}')
print()
print('Inferred members:')
for ind in sorted(inferred_students, key=lambda i: i.remainder):
    print(f'  {ind.remainder}')

In [ ]:
# How many inferred instances per class?
print(f"{'Class':<30} #instances (inferred)")
print('-' * 52)
for cls in sorted(onto.classes_in_signature(), key=lambda c: c.remainder):
    count = len(list(reasoner.instances(cls, direct=False)))
    if count > 0:
        print(f'  {cls.remainder:<28} {count}')

In [ ]:
# Infer all types of a specific individual
from owlapy.owl_individual import OWLNamedIndividual

ind = OWLNamedIndividual(IRI(BASE_IRI, 'graduate_student_01'))

inferred_types = list(reasoner.types(ind, direct=False))
print('Inferred types of graduate_student_01:')
for t in sorted(inferred_types, key=lambda c: c.remainder):
    print(f'  {t.remainder}')

In [ ]:
# Object property values of an individual (courses taken by graduate_student_03)
from owlapy.owl_property import OWLObjectProperty

takes_course = OWLObjectProperty(IRI(BASE_IRI, 'takesCourse'))
ind03 = OWLNamedIndividual(IRI(BASE_IRI, 'graduate_student_03'))

values = list(reasoner.object_property_values(ind03, takes_course))
print(f'Courses taken by graduate_student_03 ({len(values)}):')
for v in sorted(values, key=lambda i: i.remainder):
    print(f'  {v.remainder}')

## 9. NL Query Generation Pipeline

We use the reasoner as a **ground-truth oracle** to automatically generate
natural-language questions with verified answers. Three reasoning types are covered:
- **SubClassOf** (direct vs multi-hop inferred)
- **Instance-of** (stated vs inferred via OWL semantics)
- **Transitive property** (`subOrganizationOf` with a 3-node chain)

In [ ]:
import random
random.seed(42)

def label(c):
    return c.remainder if hasattr(c, 'remainder') else None

named = [c for c in onto.classes_in_signature()]
names_set = {label(c) for c in named if label(c)}

# Build superclass maps for every named class
all_supers    = {}  # name → all inferred superclasses
direct_supers = {}  # name → direct superclasses only
for cls in named:
    n = label(cls)
    if not n:
        continue
    all_supers[n] = {
        label(s) for s in reasoner.super_classes(cls, direct=False)
        if label(s) and label(s) not in ('Thing', 'Nothing')
    }
    direct_supers[n] = {
        label(s) for s in reasoner.super_classes(cls, direct=True)
        if label(s) and label(s) not in ('Thing', 'Nothing')
    }

qa_subclass = []

# Positives — direct
for sub, supers in direct_supers.items():
    for sup in supers:
        qa_subclass.append({
            'question': f'Is {sub} a subclass of {sup}?',
            'answer': True,
            'reasoning_type': 'subClassOf_direct',
            'sub': sub, 'sup': sup
        })

# Positives — inferred only (not direct)
for sub, supers in all_supers.items():
    for sup in supers - direct_supers.get(sub, set()):
        qa_subclass.append({
            'question': f'Is {sub} a subclass of {sup}?',
            'answer': True,
            'reasoning_type': 'subClassOf_inferred',
            'sub': sub, 'sup': sup
        })

# Negatives — pairs with no subsumption in either direction
negatives = []
name_list = sorted(names_set)
for sub in name_list:
    for sup in name_list:
        if sub == sup:
            continue
        if sup not in all_supers.get(sub, set()) and sub not in all_supers.get(sup, set()):
            negatives.append({
                'question': f'Is {sub} a subclass of {sup}?',
                'answer': False,
                'reasoning_type': 'subClassOf_negative',
                'sub': sub, 'sup': sup
            })

n_pos = len(qa_subclass)
qa_subclass += random.sample(negatives, min(n_pos, len(negatives)))

print(f'SubClassOf Q&A  total: {len(qa_subclass)}')
for rt in ['subClassOf_direct', 'subClassOf_inferred', 'subClassOf_negative']:
    print(f'  {rt}: {sum(1 for q in qa_subclass if q["reasoning_type"]==rt)}')

In [ ]:
qa_subclass

In [ ]:
## possibile categorizzazione a livelli: direct = level1, inferred = level2
for rt, label_str in [
    ('subClassOf_direct',   'DIRECT (stated)'),
    ('subClassOf_inferred', 'INFERRED (multi-hop)'),
    ('subClassOf_negative', 'NEGATIVE'),
]:
    samples = [q for q in qa_subclass if q['reasoning_type'] == rt][:4]
    print(f'=== {label_str} ===')
    for q in samples:
        print(f"  Q: {q['question']:<50s}  A: {q['answer']}")
    print()

In [ ]:
qa_instance = []

for ind in onto.individuals_in_signature():
    if not hasattr(ind, 'remainder'):
        continue
    all_types = {
        label(t) for t in reasoner.types(ind, direct=False)
        if label(t) and label(t) not in ('Thing', 'Nothing')
    }
    direct_types = {
        label(t) for t in reasoner.types(ind, direct=True)
        if label(t) and label(t) not in ('Thing', 'Nothing')
    }

    for t in direct_types:
        qa_instance.append({
            'question': f'Is {ind.remainder} an instance of {t}?',
            'answer': True,
            'reasoning_type': 'instanceOf_direct',
            'individual': ind.remainder, 'class': t
        })
    for t in all_types - direct_types:
        qa_instance.append({
            'question': f'Is {ind.remainder} an instance of {t}?',
            'answer': True,
            'reasoning_type': 'instanceOf_inferred',
            'individual': ind.remainder, 'class': t
        })
    non_types = [n for n in names_set if n not in all_types]
    for t in random.sample(non_types, min(3, len(non_types))):
        qa_instance.append({
            'question': f'Is {ind.remainder} an instance of {t}?',
            'answer': False,
            'reasoning_type': 'instanceOf_negative',
            'individual': ind.remainder, 'class': t
        })

print(f'Instance-of Q&A  total: {len(qa_instance)}')
for rt in ['instanceOf_direct', 'instanceOf_inferred', 'instanceOf_negative']:
    print(f'  {rt}: {sum(1 for q in qa_instance if q["reasoning_type"]==rt)}')
print()
for q in qa_instance:
    print(f"  [{q['reasoning_type']:22s}]  {q['question']:<55s}  A: {q['answer']}")

In [ ]:
# subOrganizationOf is declared owl:TransitiveProperty in the ontology.
# We inject a 3-node chain into a fresh copy of the ontology and verify
# that HermiT infers the transitive link that is never explicitly stated.

from owlapy.owl_axiom import OWLClassAssertionAxiom, OWLObjectPropertyAssertionAxiom
from owlapy.owl_property import OWLObjectProperty
from owlapy.owl_individual import OWLNamedIndividual

trans_onto = SyncOntology(ONTO_PATH)

rg    = OWLNamedIndividual(IRI(BASE_IRI, 'TestResearchGroup'))
dept  = OWLNamedIndividual(IRI(BASE_IRI, 'TestDepartment'))
univ  = OWLNamedIndividual(IRI(BASE_IRI, 'TestUniversity'))
sub_org = OWLObjectProperty(IRI(BASE_IRI, 'subOrganizationOf'))

# Class assertions
trans_onto.add_axiom(OWLClassAssertionAxiom(rg,   make_class('ResearchGroup')))
trans_onto.add_axiom(OWLClassAssertionAxiom(dept, make_class('Department')))
trans_onto.add_axiom(OWLClassAssertionAxiom(univ, make_class('University')))

# Stated property chain: rg → dept → univ  (rg → univ is NOT stated)
trans_onto.add_axiom(OWLObjectPropertyAssertionAxiom(rg,   sub_org, dept))
trans_onto.add_axiom(OWLObjectPropertyAssertionAxiom(dept, sub_org, univ))

trans_reasoner = SyncReasoner(ontology=trans_onto, reasoner='HermiT')

# Query all subOrganizationOf values for rg (includes transitively inferred)
values = list(trans_reasoner.object_property_values(rg, sub_org))
print('Inferred subOrganizationOf values for TestResearchGroup:')
for v in values:
    print(f'  {v.remainder}')

# Build Q&A for transitivity
qa_transitive = []
stated_links   = [('TestResearchGroup', 'TestDepartment'),
                  ('TestDepartment',    'TestUniversity')]
inferred_links = [('TestResearchGroup', 'TestUniversity')]   # transitively inferred
negative_links = [('TestUniversity',    'TestResearchGroup'),
                  ('TestDepartment',    'TestResearchGroup')]

for s, o in stated_links:
    qa_transitive.append({
        'question': f'Is {s} a subOrganization of {o}?',
        'answer': True, 'reasoning_type': 'transitive_stated'
    })
for s, o in inferred_links:
    qa_transitive.append({
        'question': f'Is {s} a subOrganization of {o}?',
        'answer': True, 'reasoning_type': 'transitive_inferred'
    })
for s, o in negative_links:
    qa_transitive.append({
        'question': f'Is {s} a subOrganization of {o}?',
        'answer': False, 'reasoning_type': 'transitive_negative'
    })

print()
print('Transitive property Q&A:')
for q in qa_transitive:
    print(f"  [{q['reasoning_type']:20s}]  {q['question']:<55s}  A: {q['answer']}")

In [ ]:
import json as _json

all_qa = qa_subclass + qa_instance + qa_transitive

print(f'Total Q&A pairs: {len(all_qa)}')
print()
counts = {}
for q in all_qa:
    counts[q['reasoning_type']] = counts.get(q['reasoning_type'], 0) + 1
for rt, n in sorted(counts.items()):
    print(f'  {rt:<30s} {n}')

out_path = 'owlapy_generated_qa.json'
with open(out_path, 'w') as f:
    _json.dump(all_qa, f, indent=2)
print(f'\nSaved to {out_path}')

## 10. LUBM Extended: Domain/Range, EquivalentClass, Inverse Properties

Reuses `onto` and `reasoner` (univ-bench-inst.ttl) from the top of the notebook.
Each sub-section extracts ground-truth facts from the TBox/reasoner and generates Q&A pairs.

In [ ]:
from owlapy.owl_axiom import OWLObjectPropertyDomainAxiom, OWLObjectPropertyRangeAxiom

SKIP = {'Thing', 'Nothing'}
domain_map, range_map = {}, {}

for prop in onto.object_properties_in_signature():
    for axiom in onto.object_property_domain_axioms(prop):
        d = axiom.get_domain()
        if hasattr(d,'remainder') and d.remainder not in SKIP:
            domain_map[prop.remainder] = d.remainder
    for axiom in onto.object_property_range_axioms(prop):
        r = axiom.get_range()
        if hasattr(r,'remainder') and r.remainder not in SKIP:
            range_map[prop.remainder] = r.remainder

all_cls_names = sorted(names_set)
qa_domain_range = []

for prop, dom in domain_map.items():
    qa_domain_range.append({'question': f'Is {dom} the domain of {prop}?',
                             'answer': True,  'reasoning_type': 'domain_true',  'property': prop})
    wrong = random.choice([c for c in all_cls_names if c != dom])
    qa_domain_range.append({'question': f'Is {wrong} the domain of {prop}?',
                             'answer': False, 'reasoning_type': 'domain_false', 'property': prop})

for prop, rng in range_map.items():
    qa_domain_range.append({'question': f'Is {rng} the range of {prop}?',
                             'answer': True,  'reasoning_type': 'range_true',  'property': prop})
    wrong = random.choice([c for c in all_cls_names if c != rng])
    qa_domain_range.append({'question': f'Is {wrong} the range of {prop}?',
                             'answer': False, 'reasoning_type': 'range_false', 'property': prop})

print(f'Domain/Range Q&A: {len(qa_domain_range)}')
for q in [q for q in qa_domain_range if q['answer']]:
    print(f"  {q['question']:<55s}  → {q['answer']}")

In [ ]:
from owlapy.owl_individual import OWLNamedIndividual
from owlapy.owl_property import OWLObjectProperty
from owlapy.owl_axiom import OWLClassAssertionAxiom, OWLObjectPropertyAssertionAxiom

# Domain inference: an individual that uses teacherOf (domain=Faculty) is inferred to be Faculty
dom_demo = SyncOntology(ONTO_PATH)
unk_person  = OWLNamedIndividual(IRI(BASE_IRI, 'UnknownPerson'))
some_course = OWLNamedIndividual(IRI(BASE_IRI, 'SomeCourse'))
teacher_of  = OWLObjectProperty(IRI(BASE_IRI, 'teacherOf'))

dom_demo.add_axiom(OWLClassAssertionAxiom(some_course, make_class('Course')))
dom_demo.add_axiom(OWLObjectPropertyAssertionAxiom(unk_person, teacher_of, some_course))

dom_demo_r = SyncReasoner(ontology=dom_demo, reasoner='HermiT')
inf_types = {label(t) for t in dom_demo_r.types(unk_person, direct=False) if label(t) and label(t) not in SKIP}

print('UnknownPerson — only stated: teacherOf SomeCourse')
print('Inferred types:', sorted(inf_types))

qa_domain_range += [
    {'question': 'If X teacherOf Y, must X be a Faculty?',
     'answer': 'Faculty' in inf_types, 'reasoning_type': 'domain_inference'},
    {'question': 'If X teacherOf Y, must X be a Person?',
     'answer': 'Person'  in inf_types, 'reasoning_type': 'domain_inference'},
    {'question': 'If X teacherOf Y, must X be a Student?',
     'answer': 'Student' in inf_types, 'reasoning_type': 'domain_inference'},
    {'question': 'If X advisor Y, must Y be a Professor?',
     'answer': True, 'reasoning_type': 'range_inference'},
]
print('\nDomain inference Q&A added.')


In [ ]:
import json as _json

print(f'Domain/Range Q&A pairs: {len(qa_domain_range)}')
counts_dr = {}
for q in qa_domain_range:
    counts_dr[q['reasoning_type']] = counts_dr.get(q['reasoning_type'], 0) + 1
for rt, n in sorted(counts_dr.items()):
    print(f'  {rt:<30s} {n}')

out_dr = 'owlapy_generated_qa_domain_range.json'
with open(out_dr, 'w') as f:
    _json.dump(qa_domain_range, f, indent=2)
print(f'Saved to {out_dr}')


In [ ]:
# equivalent_classes_axioms(cls) avoids the get_tbox_axioms mapper bug
# because it only converts axioms for one specific class at a time.

stated_equiv = set()
for cls in onto.classes_in_signature():
    try:
        for axiom in onto.equivalent_classes_axioms(cls):
            try:
                named = list(axiom.named_classes)
                for i, a in enumerate(named):
                    for b in named[i+1:]:
                        if label(a) and label(b):
                            stated_equiv.add(tuple(sorted([label(a), label(b)])))
            except AssertionError:
                pass
    except AssertionError:
        pass

inferred_equiv = set()
for cls in onto.classes_in_signature():
    n = label(cls)
    if not n:
        continue
    for e in reasoner.equivalent_classes(cls):
        e_name = label(e)
        if e_name and e_name != n and e_name not in SKIP:
            inferred_equiv.add(tuple(sorted([n, e_name])))

all_equiv = stated_equiv | inferred_equiv

qa_equiv = []
for a, b in sorted(all_equiv):
    qa_equiv.append({'question': f'Is {a} equivalent to {b}?',
                     'answer': True, 'reasoning_type': 'equiv_pos'})

non_equiv = [(a, b) for a in sorted(names_set) for b in sorted(names_set)
             if a < b and tuple(sorted([a,b])) not in all_equiv]
for a, b in random.sample(non_equiv, min(len(qa_equiv)*2, len(non_equiv))):
    qa_equiv.append({'question': f'Is {a} equivalent to {b}?',
                     'answer': False, 'reasoning_type': 'equiv_neg'})

print(f'EquivalentClass Q&A: {len(qa_equiv)}')
print(f'  positives: {sum(1 for q in qa_equiv if q["answer"])}')
print(f'  negatives: {sum(1 for q in qa_equiv if not q["answer"])}')
print()
for q in [q for q in qa_equiv if q['answer']]:
    print(f"  ✓ {q['question']}")

In [ ]:
from owlapy.owl_axiom import OWLInverseObjectPropertiesAxiom
import random 
# get_rbox_axioms() contains property axioms (no complex class expressions → no mapper bug)
inverse_pairs_lubm = []
for axiom in onto.get_rbox_axioms():
    try:
        if isinstance(axiom, OWLInverseObjectPropertiesAxiom):
            p1, p2 = axiom.get_first_property(), axiom.get_second_property()
            if hasattr(p1,'remainder') and hasattr(p2,'remainder'):
                inverse_pairs_lubm.append((p1.remainder, p2.remainder))
    except AssertionError:
        pass

print(f'Inverse pairs: {len(inverse_pairs_lubm)}')
for p1, p2 in inverse_pairs_lubm:
    print(f'  {p1}  inverseOf  {p2}')

inv_set = {tuple(sorted([p1,p2])) for p1,p2 in inverse_pairs_lubm}
all_prop_names = {p.remainder for p in onto.object_properties_in_signature() if hasattr(p,'remainder')}

qa_inverse = []
for p1, p2 in inverse_pairs_lubm:
    qa_inverse.append({'question': f'Is {p1} the inverse of {p2}?',
                       'answer': True, 'reasoning_type': 'inverse_pos'})

neg_inv = [(a,b) for a in sorted(all_prop_names) for b in sorted(all_prop_names)
           if a < b and tuple(sorted([a,b])) not in inv_set]
for a, b in random.sample(neg_inv, min(len(inverse_pairs_lubm)*3, len(neg_inv))):
    qa_inverse.append({'question': f'Is {a} the inverse of {b}?',
                       'answer': False, 'reasoning_type': 'inverse_neg'})

if inverse_pairs_lubm:
    p1_name, p2_name = inverse_pairs_lubm[0]
    inv_demo = SyncOntology(ONTO_PATH)
    ind_a = OWLNamedIndividual(IRI(BASE_IRI, 'InvDemoA'))
    ind_b = OWLNamedIndividual(IRI(BASE_IRI, 'InvDemoB'))
    prop1  = OWLObjectProperty(IRI(BASE_IRI, p1_name))
    prop2  = OWLObjectProperty(IRI(BASE_IRI, p2_name))
    inv_demo.add_axiom(OWLObjectPropertyAssertionAxiom(ind_a, prop1, ind_b))
    inv_demo_r = SyncReasoner(ontology=inv_demo, reasoner='HermiT')
    vals = [label(v) for v in inv_demo_r.object_property_values(ind_b, prop2)]
    inferred = 'InvDemoA' in vals
    print(f'\nDemo: stated {p1_name}(InvDemoA, InvDemoB)')
    print(f'Inferred {p2_name}(InvDemoB, InvDemoA): {inferred}')
    qa_inverse.append({'question': f'If X {p1_name} Y, is Y {p2_name} X?',
                       'answer': inferred, 'reasoning_type': 'inverse_inference'})

print(f'\nInverse Q&A total: {len(qa_inverse)}')

## 11. Pizza Ontology: Functional, InverseFunctional, Transitive, Cardinality, Disjoint, Symmetric

Uses `data/pizza.ttl` for OWL concepts absent from LUBM.
Each demo creates a fresh copy of the ontology, adds minimal ABox data, and queries HermiT.

In [ ]:
PIZZA_PATH = 'data/pizza.ttl'
PIZZA_IRI  = 'http://www.co-ode.org/ontologies/pizza/pizza.owl#'

pizza_onto     = SyncOntology(PIZZA_PATH)
pizza_reasoner = SyncReasoner(ontology=PIZZA_PATH, reasoner='HermiT')

print('Pizza ontology loaded. Consistent:', pizza_reasoner.has_consistent_ontology())
print(f'Classes:           {len(list(pizza_onto.classes_in_signature()))}')
print(f'Object properties: {len(list(pizza_onto.object_properties_in_signature()))}')

def pizza_cls(name):  return OWLClass(IRI(PIZZA_IRI, name))
def pizza_ind(name):  return OWLNamedIndividual(IRI(PIZZA_IRI, name))
def pizza_prop(name): return OWLObjectProperty(IRI(PIZZA_IRI, name))
def plabel(c):        return c.remainder if hasattr(c, 'remainder') else None

In [ ]:
from owlapy.owl_axiom import (
    OWLFunctionalObjectPropertyAxiom,
    OWLInverseFunctionalObjectPropertyAxiom,
    OWLTransitiveObjectPropertyAxiom,
    OWLSymmetricObjectPropertyAxiom,
    OWLInverseObjectPropertiesAxiom,
)

# get_rbox_axioms() contains only property axioms — no class expressions → no mapper bug
char_map = {'functional': [], 'inv_functional': [], 'transitive': [], 'symmetric': []}
inverse_pairs_pizza = []

for axiom in pizza_onto.get_rbox_axioms():
    try:
        if isinstance(axiom, OWLFunctionalObjectPropertyAxiom):
            p = axiom.get_property()
            if hasattr(p,'remainder'): char_map['functional'].append(p.remainder)
        elif isinstance(axiom, OWLInverseFunctionalObjectPropertyAxiom):
            p = axiom.get_property()
            if hasattr(p,'remainder'): char_map['inv_functional'].append(p.remainder)
        elif isinstance(axiom, OWLTransitiveObjectPropertyAxiom):
            p = axiom.get_property()
            if hasattr(p,'remainder'): char_map['transitive'].append(p.remainder)
        elif isinstance(axiom, OWLSymmetricObjectPropertyAxiom):
            p = axiom.get_property()
            if hasattr(p,'remainder'): char_map['symmetric'].append(p.remainder)
        elif isinstance(axiom, OWLInverseObjectPropertiesAxiom):
            p1, p2 = axiom.get_first_property(), axiom.get_second_property()
            if hasattr(p1,'remainder') and hasattr(p2,'remainder'):
                inverse_pairs_pizza.append((p1.remainder, p2.remainder))
    except AssertionError:
        pass

for char, props in char_map.items():
    print(f'{char:20s}: {props}')
print(f'inverse pairs       : {inverse_pairs_pizza}')

In [ ]:
all_pizza_props = sorted({p.remainder for p in pizza_onto.object_properties_in_signature()
                           if hasattr(p,'remainder')})
import random

qa_prop_chars = []
for prop in all_pizza_props:
    for char, char_list in char_map.items():
        qa_prop_chars.append({
            'question': f'Is {prop} a {char.replace("_"," ")} property?',
            'answer': prop in char_list,
            'reasoning_type': f'property_char_{char}',
            'property': prop, 'characteristic': char
        })

# Balance: keep all trues + same number of falses
true_q  = [q for q in qa_prop_chars if q['answer']]
false_q = random.sample([q for q in qa_prop_chars if not q['answer']],
                         min(len(true_q)*2, sum(1 for q in qa_prop_chars if not q['answer'])))
qa_prop_chars = true_q + false_q

print(f'Property characteristics Q&A: {len(qa_prop_chars)}')
print(f'  True:  {sum(1 for q in qa_prop_chars if q["answer"])}')
print(f'  False: {sum(1 for q in qa_prop_chars if not q["answer"])}')
print('\nTrue statements:')
for q in true_q:
    print(f"  ✓ {q['question']}")

In [ ]:
# hasBase is functional: if Pizza hasBase A and Pizza hasBase B, HermiT infers A sameAs B
func_demo = SyncOntology(PIZZA_PATH)
f_pizza = pizza_ind('FuncPizza')
f_base_a = pizza_ind('FuncBaseA')
f_base_b = pizza_ind('FuncBaseB')
has_base = pizza_prop('hasBase')

func_demo.add_axiom(OWLClassAssertionAxiom(f_pizza,  pizza_cls('Pizza')))
func_demo.add_axiom(OWLClassAssertionAxiom(f_base_a, pizza_cls('PizzaBase')))
func_demo.add_axiom(OWLClassAssertionAxiom(f_base_b, pizza_cls('PizzaBase')))
func_demo.add_axiom(OWLObjectPropertyAssertionAxiom(f_pizza, has_base, f_base_a))
func_demo.add_axiom(OWLObjectPropertyAssertionAxiom(f_pizza, has_base, f_base_b))

func_r   = SyncReasoner(ontology=func_demo, reasoner='HermiT')
same_a   = [plabel(i) for i in func_r.same_individuals(f_base_a)]
print(f'same_individuals(FuncBaseA): {same_a}')

func_inferred = 'FuncBaseB' in same_a
print(f'\nQ: If hasBase is functional and a pizza has two bases, are they the same individual?')
print(f'A: {func_inferred}')

In [ ]:
# hasTopping is inverse functional:
# if PizzaA hasTopping T and PizzaB hasTopping T (same individual), HermiT infers PizzaA sameAs PizzaB
ifunc_demo = SyncOntology(PIZZA_PATH)
if_pizza_a  = pizza_ind('IFPizzaA')
if_pizza_b  = pizza_ind('IFPizzaB')
if_topping  = pizza_ind('IFSharedTopping')
has_topping = pizza_prop('hasTopping')

ifunc_demo.add_axiom(OWLClassAssertionAxiom(if_pizza_a, pizza_cls('Pizza')))
ifunc_demo.add_axiom(OWLClassAssertionAxiom(if_pizza_b, pizza_cls('Pizza')))
ifunc_demo.add_axiom(OWLClassAssertionAxiom(if_topping, pizza_cls('PizzaTopping')))
ifunc_demo.add_axiom(OWLObjectPropertyAssertionAxiom(if_pizza_a, has_topping, if_topping))
ifunc_demo.add_axiom(OWLObjectPropertyAssertionAxiom(if_pizza_b, has_topping, if_topping))

ifunc_r      = SyncReasoner(ontology=ifunc_demo, reasoner='HermiT')
same_pizza_a = [plabel(i) for i in ifunc_r.same_individuals(if_pizza_a)]
print(f'same_individuals(IFPizzaA): {same_pizza_a}')

ifunc_inferred = 'IFPizzaB' in same_pizza_a
print(f'\nQ: If hasTopping is inverse functional and IFPizzaA and IFPizzaB both hasTopping the same individual, are they the same pizza?')
print(f'A: {ifunc_inferred}')

In [ ]:
# isIngredientOf is transitive
# Chain: ToppingA isIngredientOf SauceB, SauceB isIngredientOf PizzaC
# → ToppingA isIngredientOf PizzaC inferred
trans_pizza = SyncOntology(PIZZA_PATH)
tr_topping  = pizza_ind('TransToppingA')
tr_sauce    = pizza_ind('TransSauceB')
tr_pizza    = pizza_ind('TransPizzaC')
is_ingr_of  = pizza_prop('isIngredientOf')

trans_pizza.add_axiom(OWLClassAssertionAxiom(tr_topping, pizza_cls('PizzaTopping')))
trans_pizza.add_axiom(OWLClassAssertionAxiom(tr_sauce,   pizza_cls('PizzaTopping')))
trans_pizza.add_axiom(OWLClassAssertionAxiom(tr_pizza,   pizza_cls('Pizza')))
trans_pizza.add_axiom(OWLObjectPropertyAssertionAxiom(tr_topping, is_ingr_of, tr_sauce))
trans_pizza.add_axiom(OWLObjectPropertyAssertionAxiom(tr_sauce,   is_ingr_of, tr_pizza))

trans_pizza_r = SyncReasoner(ontology=trans_pizza, reasoner='HermiT')
tr_vals = [plabel(v) for v in trans_pizza_r.object_property_values(tr_topping, is_ingr_of)]
print(f'isIngredientOf values for TransToppingA: {tr_vals}')

trans_inferred = 'TransPizzaC' in tr_vals
print(f'\nQ: If isIngredientOf is transitive and A isIngredientOf B and B isIngredientOf C, is A isIngredientOf C?')
print(f'A: {trans_inferred}')

In [ ]:
# InterestingPizza ≡ Pizza ∧ (≥3 hasTopping)
# Test 1: 3 distinct toppings → IS classified as InterestingPizza
# Test 2: 2 distinct toppings → NOT classified (OWA: absence of info ≠ negation)
from owlapy.owl_axiom import OWLDifferentIndividualsAxiom

card_demo = SyncOntology(PIZZA_PATH)
has_top   = pizza_prop('hasTopping')
top_cls   = pizza_cls('PizzaTopping')

rich  = pizza_ind('RichPizza')
t1, t2, t3 = pizza_ind('RichT1'), pizza_ind('RichT2'), pizza_ind('RichT3')
card_demo.add_axiom(OWLClassAssertionAxiom(rich, pizza_cls('Pizza')))
for t in [t1, t2, t3]:
    card_demo.add_axiom(OWLClassAssertionAxiom(t, top_cls))
    card_demo.add_axiom(OWLObjectPropertyAssertionAxiom(rich, has_top, t))
card_demo.add_axiom(OWLDifferentIndividualsAxiom([t1, t2, t3]))

poor  = pizza_ind('PoorPizza')
p1, p2 = pizza_ind('PoorT1'), pizza_ind('PoorT2')
card_demo.add_axiom(OWLClassAssertionAxiom(poor, pizza_cls('Pizza')))
for t in [p1, p2]:
    card_demo.add_axiom(OWLClassAssertionAxiom(t, top_cls))
    card_demo.add_axiom(OWLObjectPropertyAssertionAxiom(poor, has_top, t))
card_demo.add_axiom(OWLDifferentIndividualsAxiom([p1, p2]))

card_r = SyncReasoner(ontology=card_demo, reasoner='HermiT')

types_rich = {plabel(t) for t in card_r.types(rich, direct=False) if plabel(t) and plabel(t) != 'Thing'}
types_poor = {plabel(t) for t in card_r.types(poor, direct=False) if plabel(t) and plabel(t) != 'Thing'}

print('RichPizza types:', sorted(types_rich))
print('PoorPizza types:', sorted(types_poor))
print(f'\nQ: Is a pizza with 3 distinct toppings an InterestingPizza?  → {"InterestingPizza" in types_rich}')
print(f'Q: Is a pizza with 2 distinct toppings an InterestingPizza?  → {"InterestingPizza" in types_poor}')
print('(OWA: 2-topping pizza is not classified, but not inconsistent either)')

In [ ]:
# Use reasoner.disjoint_classes() instead of get_tbox_axioms() to avoid the mapper bug.
# This gives INFERRED disjointness (superset of stated), which is even better for Q&A.

disjoint_pizza = set()
for cls in pizza_onto.classes_in_signature():
    cn = plabel(cls)
    if not cn or cn in SKIP:
        continue
    for d in pizza_reasoner.disjoint_classes(cls):
        dn = plabel(d)
        if dn and dn not in SKIP and cn != dn:
            disjoint_pizza.add(tuple(sorted([cn, dn])))

print(f'Disjoint pairs (inferred): {len(disjoint_pizza)}')
for pair in sorted(disjoint_pizza):
    print(f'  {pair[0]}  ⊥  {pair[1]}')

qa_disjoint = []
for a, b in sorted(disjoint_pizza):
    qa_disjoint.append({'question': f'Can an individual be both {a} and {b}?',
                        'answer': False, 'reasoning_type': 'disjoint_neg'})

pizza_cls_names = {plabel(c) for c in pizza_onto.classes_in_signature() if plabel(c) and plabel(c) not in SKIP}
non_disj = [(a,b) for a in sorted(pizza_cls_names) for b in sorted(pizza_cls_names)
            if a < b and tuple(sorted([a,b])) not in disjoint_pizza]
for a, b in random.sample(non_disj, min(len(qa_disjoint), len(non_disj))):
    qa_disjoint.append({'question': f'Can an individual be both {a} and {b}?',
                        'answer': True, 'reasoning_type': 'disjoint_pos'})

# Inconsistency demo
if disjoint_pizza:
    pair = sorted(disjoint_pizza)[0]
    clash_demo = SyncOntology(PIZZA_PATH)
    clash_ind  = pizza_ind('ClashIndividual')
    clash_demo.add_axiom(OWLClassAssertionAxiom(clash_ind, pizza_cls(pair[0])))
    clash_demo.add_axiom(OWLClassAssertionAxiom(clash_ind, pizza_cls(pair[1])))
    clash_r    = SyncReasoner(ontology=clash_demo, reasoner='HermiT')
    consistent = clash_r.has_consistent_ontology()
    print(f'\nClashIndividual typed as both {pair[0]} and {pair[1]} → consistent: {consistent}')
    print(f'Q: Does asserting an individual of two disjoint classes cause inconsistency?  → {not consistent}')

print(f'\nDisjoint Q&A: {len(qa_disjoint)}')

In [ ]:
# Pizza has no symmetric property → add hasSimilarTaste synthetically
# If PizzaX hasSimilarTaste PizzaY, then PizzaY hasSimilarTaste PizzaX (inferred)
sym_demo  = SyncOntology(PIZZA_PATH)
sym_prop  = pizza_prop('hasSimilarTaste')
sym_x     = pizza_ind('SymPizzaX')
sym_y     = pizza_ind('SymPizzaY')

sym_demo.add_axiom(OWLSymmetricObjectPropertyAxiom(sym_prop))
sym_demo.add_axiom(OWLClassAssertionAxiom(sym_x, pizza_cls('Pizza')))
sym_demo.add_axiom(OWLClassAssertionAxiom(sym_y, pizza_cls('Pizza')))
sym_demo.add_axiom(OWLObjectPropertyAssertionAxiom(sym_x, sym_prop, sym_y))

sym_r   = SyncReasoner(ontology=sym_demo, reasoner='HermiT')
sym_vals = [plabel(v) for v in sym_r.object_property_values(sym_y, sym_prop)]
print(f'hasSimilarTaste values for SymPizzaY (only stated: SymPizzaX → SymPizzaY): {sym_vals}')

sym_inferred = 'SymPizzaX' in sym_vals
print(f'\nQ: If hasSimilarTaste is symmetric and SymPizzaX hasSimilarTaste SymPizzaY, does SymPizzaY hasSimilarTaste SymPizzaX?')
print(f'A: {sym_inferred}')

In [ ]:
import json as _json

# Inference demos as Q&A entries
qa_inference_demos = [
    {'question': 'If X teacherOf Y, must X be a Faculty?',                                                                    'answer': True,  'reasoning_type': 'domain_inference',        'ontology': 'lubm'},
    {'question': 'If X advisor Y, must Y be a Professor?',                                                                    'answer': True,  'reasoning_type': 'range_inference',         'ontology': 'lubm'},
    {'question': 'If hasBase is functional and a pizza has two bases, are they the same individual?',                         'answer': True,  'reasoning_type': 'functional_inference',     'ontology': 'pizza'},
    {'question': 'If hasTopping is inverse functional and two pizzas share the same topping individual, are they the same?',  'answer': True,  'reasoning_type': 'inv_functional_inference', 'ontology': 'pizza'},
    {'question': 'If isIngredientOf is transitive and A isIngredientOf B and B isIngredientOf C, is A isIngredientOf C?',     'answer': True,  'reasoning_type': 'transitive_pizza',        'ontology': 'pizza'},
    {'question': 'Is a pizza with 3 distinct toppings an InterestingPizza?',                                                  'answer': True,  'reasoning_type': 'cardinality_pos',         'ontology': 'pizza'},
    {'question': 'Is a pizza with only 2 distinct toppings an InterestingPizza?',                                             'answer': False, 'reasoning_type': 'cardinality_neg',         'ontology': 'pizza'},
    {'question': 'If hasSimilarTaste is symmetric and X hasSimilarTaste Y, does Y hasSimilarTaste X?',                        'answer': True,  'reasoning_type': 'symmetric_inference',     'ontology': 'pizza'},
    {'question': 'Does asserting an individual of two disjoint classes make the ontology inconsistent?',                      'answer': True,  'reasoning_type': 'disjoint_inconsistency',  'ontology': 'pizza'},
]

for q in qa_domain_range + qa_equiv + qa_inverse + qa_prop_chars + qa_disjoint:
    if 'ontology' not in q:
        q['ontology'] = 'lubm' if q in qa_domain_range + qa_equiv + qa_inverse else 'pizza'

all_extended = qa_domain_range + qa_equiv + qa_inverse + qa_prop_chars + qa_disjoint + qa_inference_demos

print(f'Total extended Q&A: {len(all_extended)}')
for rt in sorted(set(q['reasoning_type'] for q in all_extended)):
    n = sum(1 for q in all_extended if q['reasoning_type'] == rt)
    print(f'  {rt:<35s} {n}')

out = 'owlapy_generated_qa_extended.json'
with open(out, 'w') as f:
    _json.dump(all_extended, f, indent=2)
print(f'\nSaved to {out}')

## 12. Proof Chain Generation

For each Q&A entry we build a **structured proof chain** — an ordered list of
logic steps (axiom lookups, transitivity applications, ABox checks) that justify
the answer. This chain can later be passed to an LLM for verbalisation.

The key insight is that owlapy already knows *why* an answer is true:
`super_classes(direct=True)` gives us the hop-by-hop path,
`types(ind, direct=True)` gives us the stated ABox type,
and `ind_object_properties` + `object_property_values` expose the relevant ABox facts.

In [ ]:
from collections import deque

# ── path finder ─────────────────────────────────────────────────────────────
def find_superclass_path(start, end, reasoner, base_iri, skip=frozenset({'Thing','Nothing'})):
    """BFS: shortest chain of direct-superclass hops from start to end."""
    queue = deque([(start, [start])])
    visited = {start}
    while queue:
        node, path = queue.popleft()
        cls = OWLClass(IRI(base_iri, node))
        for sup in reasoner.super_classes(cls, direct=True):
            sn = sup.remainder if hasattr(sup, 'remainder') else None
            if not sn or sn in skip:
                continue
            new_path = path + [sn]
            if sn == end:
                return new_path
            if sn not in visited:
                visited.add(sn)
                queue.append((sn, new_path))
    return None


# ── proof chain builder ──────────────────────────────────────────────────────
def build_proof_chain(q, onto, reasoner, base_iri,
                      skip=frozenset({'Thing', 'Nothing'})):
    """Return a list of dicts, each representing one reasoning step."""
    rtype  = q.get('reasoning_type', '')
    steps  = []
    step   = lambda t, f, s='': steps.append({'type': t, 'fact': f, 'source': s})

    # ── SubClassOf direct ────────────────────────────────────────────────────
    if rtype == 'subClassOf_direct':
        sub, sup = q['sub'], q['sup']
        step('SubClassOf_stated', f'{sub} ⊑ {sup}', 'TBox (direct axiom)')

    # ── SubClassOf inferred (multi-hop) ──────────────────────────────────────
    elif rtype == 'subClassOf_inferred':
        sub, sup = q['sub'], q['sup']
        path = find_superclass_path(sub, sup, reasoner, base_iri)
        if path:
            for i in range(len(path) - 1):
                step('SubClassOf_stated', f'{path[i]} ⊑ {path[i+1]}', 'TBox (direct axiom)')
            step('SubClassOf_transitivity',
                 f'{sub} ⊑ {sup}  (transitivity over {len(path)-1} hop(s))',
                 'OWL semantics')
        else:
            step('Hierarchy_check', f'Path {sub} → {sup} not found', 'Reasoner')

    # ── SubClassOf negative ──────────────────────────────────────────────────
    elif rtype == 'subClassOf_negative':
        sub, sup = q['sub'], q['sup']
        step('Hierarchy_check',
             f'No subclass path from {sub} to {sup} in the inferred hierarchy',
             'Reasoner (closed-world over the hierarchy)')

    # ── InstanceOf direct ────────────────────────────────────────────────────
    elif rtype == 'instanceOf_direct':
        step('ABox_type',
             f'{q["individual"]} rdf:type {q["class"]}',
             'ABox (stated)')

    # ── InstanceOf inferred ──────────────────────────────────────────────────
    elif rtype == 'instanceOf_inferred':
        ind_name, cls_name = q['individual'], q['class']
        ind = OWLNamedIndividual(IRI(base_iri, ind_name))

        # (a) stated type(s)
        direct_types = [label(t) for t in reasoner.types(ind, direct=True)
                        if label(t) and label(t) not in skip]
        for t in direct_types:
            step('ABox_type', f'{ind_name} rdf:type {t}', 'ABox (stated)')

        # (b) subclass chain from stated type to target, if reachable
        chain_found = False
        for t in direct_types:
            if t == cls_name:
                chain_found = True
                break
            path = find_superclass_path(t, cls_name, reasoner, base_iri)
            if path:
                for i in range(len(path) - 1):
                    step('SubClassOf_stated', f'{path[i]} ⊑ {path[i+1]}', 'TBox')
                step('SubClassOf_transitivity',
                     f'{t} ⊑ {cls_name}  ({len(path)-1} hop(s))',
                     'OWL semantics')
                chain_found = True
                break

        # (c) if no pure subclass path, target may be reached via equivalentClass
        if not chain_found:
            step('EquivalentClass_check',
                 f'{cls_name} has an equivalentClass definition — checking ABox facts',
                 'TBox')

            # Show object-property values of the individual
            for prop in onto.object_properties_in_signature():
                try:
                    vals = list(reasoner.object_property_values(ind, prop))
                    for v in vals:
                        if not hasattr(v, 'remainder'): continue
                        step('ABox_property',
                             f'{ind_name} {prop.remainder} {v.remainder}',
                             'ABox (stated)')
                        # type of value
                        val_ind   = OWLNamedIndividual(IRI(base_iri, v.remainder))
                        val_types = [label(t) for t in reasoner.types(val_ind, direct=True)
                                     if label(t) and label(t) not in skip]
                        for vt in val_types:
                            step('ABox_type', f'{v.remainder} rdf:type {vt}', 'ABox (stated)')
                except Exception:
                    pass

        step('EquivalentClass_satisfaction',
             f'{ind_name} satisfies all conditions of {cls_name}',
             'Reasoner (HermiT)')

    # ── Domain / Range ───────────────────────────────────────────────────────
    elif rtype in ('domain_true', 'domain_false', 'domain_inference'):
        prop = q.get('property', '?')
        val  = domain_map.get(prop, '?')
        step('Domain_axiom', f'domain({prop}) = {val}', 'TBox (stated)')
        if rtype == 'domain_inference':
            step('Domain_inference',
                 f'Any individual using {prop} must belong to {val}',
                 'OWL semantics')

    elif rtype in ('range_true', 'range_false', 'range_inference'):
        prop = q.get('property', '?')
        val  = range_map.get(prop, '?')
        step('Range_axiom', f'range({prop}) = {val}', 'TBox (stated)')

    # ── EquivalentClass ──────────────────────────────────────────────────────
    elif rtype == 'equiv_pos':
        step('EquivalentClass',
             f'{q["classA"]} ≡ {q["classB"]}',
             'TBox / Reasoner')

    elif rtype == 'equiv_neg':
        step('EquivalentClass_check',
             f'No equivalentClass relation between {q["classA"]} and {q["classB"]}',
             'Reasoner')

    # ── Inverse property ─────────────────────────────────────────────────────
    elif rtype == 'inverse_pos':
        step('InverseOf_axiom', q['question'].rstrip('?'), 'RBox (stated)')

    elif rtype == 'inverse_inference':
        step('InverseOf_axiom', 'P₁ inverseOf P₂', 'RBox (stated)')
        step('ABox_property',   'X P₁ Y',          'ABox (stated)')
        step('Inference',       'Y P₂ X — inferred via inverseOf semantics', 'OWL semantics')

    # ── Transitive property ──────────────────────────────────────────────────
    elif rtype in ('transitive_stated', 'transitive_inferred', 'transitive_pizza'):
        step('Property_axiom', 'P is owl:TransitiveProperty', 'RBox / TBox (stated)')
        if rtype != 'transitive_stated':
            step('ABox_property',  'A P B (stated)', 'ABox')
            step('ABox_property',  'B P C (stated)', 'ABox')
            step('Inference',      'A P C — inferred by transitivity (A→B, B→C ⟹ A→C)',
                 'OWL semantics')

    # ── Functional / InverseFunctional ───────────────────────────────────────
    elif rtype == 'functional_inference':
        step('Property_axiom', 'hasBase is owl:FunctionalProperty', 'RBox / TBox (stated)')
        step('ABox_property',  'Pizza hasBase A (stated)', 'ABox')
        step('ABox_property',  'Pizza hasBase B (stated)', 'ABox')
        step('Inference',
             'A sameAs B — functional property allows at most one value; two values → same individual',
             'OWL semantics')

    elif rtype == 'inv_functional_inference':
        step('Property_axiom', 'hasTopping is owl:InverseFunctionalProperty', 'RBox / TBox (stated)')
        step('ABox_property',  'PizzaA hasTopping T (stated)', 'ABox')
        step('ABox_property',  'PizzaB hasTopping T (stated)', 'ABox')
        step('Inference',
             'PizzaA sameAs PizzaB — inverse functional: same object → same subject',
             'OWL semantics')

    # ── Symmetric property ───────────────────────────────────────────────────
    elif rtype == 'symmetric_inference':
        step('Property_axiom', 'P is owl:SymmetricProperty', 'TBox (stated)')
        step('ABox_property',  'X P Y (stated)', 'ABox')
        step('Inference',      'Y P X — inferred by symmetry', 'OWL semantics')

    # ── Cardinality ──────────────────────────────────────────────────────────
    elif rtype == 'cardinality_pos':
        step('EquivalentClass', 'InterestingPizza ≡ Pizza ∧ (≥3 hasTopping)', 'TBox')
        step('ABox_count',      '3 distinct toppings asserted (owl:differentIndividuals)', 'ABox')
        step('Inference',       'minCardinality ≥ 3 satisfied → classified as InterestingPizza',
             'OWL semantics')

    elif rtype == 'cardinality_neg':
        step('EquivalentClass', 'InterestingPizza ≡ Pizza ∧ (≥3 hasTopping)', 'TBox')
        step('ABox_count',      'Only 2 distinct toppings asserted', 'ABox')
        step('OWA_note',
             'Under OWA, additional toppings may exist — reasoner cannot classify as InterestingPizza',
             'Open World Assumption')

    # ── Disjoint ─────────────────────────────────────────────────────────────
    elif rtype == 'disjoint_neg':
        step('DisjointWith',
             f'{q.get("classA","?")} owl:disjointWith {q.get("classB","?")}',
             'TBox / Reasoner')
        step('Inference',
             'No individual can satisfy both classes simultaneously',
             'OWL semantics')

    elif rtype == 'disjoint_inconsistency':
        step('DisjointWith',   'A owl:disjointWith B', 'TBox / Reasoner')
        step('ABox_type',      'X rdf:type A (stated)', 'ABox')
        step('ABox_type',      'X rdf:type B (stated)', 'ABox')
        step('Inconsistency',  'Contradiction: X cannot be in both disjoint classes → inconsistent',
             'OWL semantics')

    # ── Property characteristics (boolean) ───────────────────────────────────
    elif rtype.startswith('property_char_'):
        char = rtype.replace('property_char_', '').replace('_', ' ')
        prop = q.get('property', '?')
        if q['answer']:
            step('RBox_axiom', f'{prop} is declared {char}', 'RBox / TBox (stated)')
        else:
            step('RBox_check', f'No {char} axiom found for {prop}', 'RBox / TBox')

    # ── Fallback ─────────────────────────────────────────────────────────────
    else:
        step('Generic', q['question'], 'see reasoning_type')

    # Always append a conclusion step
    steps.append({'type': 'Conclusion',
                  'fact': f'Answer: {q["answer"]}',
                  'source': ''})
    return steps


# ── pretty printer ────────────────────────────────────────────────────────────
def format_chain(q, steps):
    lines = []

    # lines.append(f'Q: {q["question"]}')
    # lines.append(f'A: {q["answer"]}  [{q["reasoning_type"]}]')

    lines.append('─' * 65)

    for i, s in enumerate(steps):
        prefix = '  ∴' if s['type'] == 'Conclusion' else f'  {i+1}.'
        src = f'  ← {s["source"]}' if s['source'] else ''
        lines.append(f'{prefix} [{s["type"]}]  {s["fact"]}{src}')

    lines.append('─' * 65)

    return '\n'.join(lines)
print('Helper functions defined.')

### Example 1 — SubClassOf (multi-hop)

In [ ]:
q1 = {'question': 'Is AssistantProfessor a subclass of Person?',
      'answer': True, 'reasoning_type': 'subClassOf_inferred',
      'sub': 'AssistantProfessor', 'sup': 'Person'}

chain1 = build_proof_chain(q1, onto, reasoner, BASE_IRI)
print(format_chain(q1, chain1))

In [ ]:
format_chain(q1, chain1)


In [ ]:
q1['question']

In [ ]:
with open("univ-bench-inst.txt") as f: 
    lubm_onto_w_inst = f.read()
print(lubm_onto_w_inst)

In [ ]:
## esempio di gt ottenuta con e senza reasoning trace
## exp senza query
import ollama

def generate_ollama_ans(model, system_prompt, prompt, **kwargs):
    messages = []
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})
    messages.append({'role': 'user', 'content': prompt})
    
    # 2. Raccogli i parametri aggiuntivi (es. temperature, top_p) in 'options'
    # Usando **kwargs, qualsiasi parametro extra inserito nella funzione finirà qui automaticamente
    options = kwargs
    
    response = ollama.chat(
        model=model,
        messages=messages,
        options=options
    )
    
    # 4. Estrai e restituisci solo il contenuto testuale della risposta
    return response['message']['content']


system_prompt = """You are an expert in semantic web in reading DL ontologies in OWL. 
Carefully read and understand the relationship stated in the context provided. 
Your goal is to create golden standard reasoning trace for the query passed by the user. 
You will be provided the original query and the steps inferred by the reasoner for correctly solving it. 
You need to explain natural language the reason and the logic needed for inferring the correct answer. 
Base your answer exclusively on the provided trace and the ontology. 
Ontology:
{onto_context}
"""
full_system_prompt  = system_prompt.format(onto_context = lubm_onto_w_inst)
_prompt = """
Original query: {query},
Correct answer to the query : {correct_answer}.
This query needs this type of reasoning to be correctly solved: {reas_type}
Reasoning trace:
{reas_trace}
"""
_prompt = _prompt.format(query = q1['question'], 
                         correct_answer = q1["answer"],
                         reas_type = q1["reasoning_type"],
                         reas_trace = format_chain(q1, chain1)
)

generated_ans = generate_ollama_ans(model="gpt-oss:20b",
                    system_prompt=full_system_prompt,
                    prompt=_prompt,
                    )

In [ ]:
print("generated ollama ans: \n", generated_ans)

In [ ]:
# Optional commented OpenRouter example omitted from collaboration copy.


### Example 2 — InstanceOf inferred via equivalentClass

In [ ]:
q2 = {'question': 'Is graduate_student_01 an instance of Student?',
      'answer': True, 'reasoning_type': 'instanceOf_inferred',
      'individual': 'graduate_student_01', 'class': 'Student'}

chain2 = build_proof_chain(q2, onto, reasoner, BASE_IRI)
format_chain(q2, chain2)

### Example 3 — Domain inference

In [ ]:
q3 = {'question': 'If X teacherOf Y, must X be a Faculty?',
      'answer': True, 'reasoning_type': 'domain_inference', 'property': 'teacherOf'}

chain3 = build_proof_chain(q3, onto, reasoner, BASE_IRI)
format_chain(q3, chain3)

### Example 4 — Transitive property

In [ ]:
q4 = {'question': 'If A subOrganizationOf B and B subOrganizationOf C, is A subOrganizationOf C?',
      'answer': True, 'reasoning_type': 'transitive_inferred'}

chain4 = build_proof_chain(q4, onto, reasoner, BASE_IRI)
format_chain(q4, chain4)

### Example 5 — Negative answer (disjoint classes)

In [ ]:
q5 = {'question': 'Can an individual be both DeepPanBase and ThinAndCrispyBase?',
      'answer': False, 'reasoning_type': 'disjoint_neg',
      'classA': 'DeepPanBase', 'classB': 'ThinAndCrispyBase'}

chain5 = build_proof_chain(q5, onto, reasoner, BASE_IRI)
format_chain(q5, chain5)

### Apply to the full Q&A dataset

In [ ]:
import json as _json

# Load the previously generated Q&A
with open('owlapy_generated_qa.json') as f:
    base_qa = _json.load(f)
with open('owlapy_generated_qa_extended.json') as f:
    extended_qa = _json.load(f)

all_qa = base_qa + extended_qa

# Attach proof chain to each entry
skipped = 0
for q in all_qa:
    try:
        q['proof_chain'] = build_proof_chain(q, onto, reasoner, BASE_IRI)
    except Exception as e:
        q['proof_chain'] = [{'type': 'Error', 'fact': str(e), 'source': ''}]
        skipped += 1

print(f'Processed: {len(all_qa)}  |  errors: {skipped}')
print(f'Avg steps per chain: {sum(len(q["proof_chain"]) for q in all_qa) / len(all_qa):.1f}')

# Stats by reasoning type
print('\nAvg chain length by reasoning_type:')
from collections import defaultdict
by_type = defaultdict(list)
for q in all_qa:
    by_type[q['reasoning_type']].append(len(q['proof_chain']))
for rt in sorted(by_type):
    vals = by_type[rt]
    print(f'  {rt:<35s} avg={sum(vals)/len(vals):.1f}  n={len(vals)}')

out = 'owlapy_generated_qa_with_chains.json'
with open(out, 'w') as f:
    _json.dump(all_qa, f, indent=2)
print(f'\nSaved to {out}')

# Show one full entry
print('\n── Sample entry (instanceOf_inferred) ──')
sample = next(q for q in all_qa if q['reasoning_type'] == 'instanceOf_inferred')
print(_json.dumps(sample, indent=2))